<a href="https://colab.research.google.com/github/aamish007/23-CS-006-CS318-DL-Lab/blob/main/Experiment_10_DL_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install torch --quiet

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import requests

In [3]:
url = "https://ocw.mit.edu/ans7870/6/6.006/s08/lecturenotes/files/t8.shakespeare.txt"

text = requests.get(url).text

print(text[:500])
print("Length:", len(text))

This is the 100th Etext file presented by Project Gutenberg, and
is presented in cooperation with World Library, Inc., from their
Library of the Future and Shakespeare CDROMS.  Project Gutenberg
often releases Etexts that are NOT placed in the Public Domain!!

Shakespeare

*This Etext has certain copyright implications you should read!*

<<THIS ELECTRONIC VERSION OF THE COMPLETE WORKS OF WILLIAM
SHAKESPEARE IS COPYRIGHT 1990-1993 BY WORLD LIBRARY, INC., AND IS
PROVIDED BY PROJECT GUTENBERG ETEXT
Length: 5458199


In [4]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)

print("Vocab size:", vocab_size)

Vocab size: 91


In [5]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [6]:
batch_size = 32
block_size = 64
n_embd = 128
n_head = 4
n_layer = 2
dropout = 0.1

device = "cuda" if torch.cuda.is_available() else "cpu"

In [7]:
def get_batch(split):
    data = train_data if split == "train" else val_data

    ix = torch.randint(len(data) - block_size, (batch_size,))

    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    return x.to(device), y.to(device)

In [8]:
class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(x.device)

In [9]:
class MultiHeadAttention(nn.Module):

    def __init__(self, embed_size, heads):
        super().__init__()

        self.embed_size = embed_size
        self.heads = heads
        self.head_dim = embed_size // heads

        self.values = nn.Linear(embed_size, embed_size)
        self.keys = nn.Linear(embed_size, embed_size)
        self.queries = nn.Linear(embed_size, embed_size)

        self.fc_out = nn.Linear(embed_size, embed_size)

    def forward(self, value, key, query, mask=None):

        N = query.shape[0]

        values = self.values(value)
        keys = self.keys(key)
        queries = self.queries(query)

        values = values.view(N, -1, self.heads, self.head_dim)
        keys = keys.view(N, -1, self.heads, self.head_dim)
        queries = queries.view(N, -1, self.heads, self.head_dim)

        energy = torch.einsum("nqhd,nkhd->nhqk", [queries, keys])

        if mask is not None:
            energy = energy.masked_fill(mask == 0, float("-1e20"))

        attention = torch.softmax(energy / math.sqrt(self.head_dim), dim=3)

        out = torch.einsum("nhql,nlhd->nqhd", [attention, values])

        out = out.reshape(N, -1, self.embed_size)

        out = self.fc_out(out)

        return out

In [10]:
class FeedForward(nn.Module):

    def __init__(self, embed_size):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(embed_size, 4*embed_size),
            nn.ReLU(),
            nn.Linear(4*embed_size, embed_size)
        )

    def forward(self, x):
        return self.net(x)

In [11]:
class EncoderBlock(nn.Module):

    def __init__(self, embed_size, heads, dropout):
        super().__init__()

        self.attention = MultiHeadAttention(embed_size, heads)

        self.norm1 = nn.LayerNorm(embed_size)
        self.norm2 = nn.LayerNorm(embed_size)

        self.feed_forward = FeedForward(embed_size)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        attention = self.attention(x, x, x)

        x = self.dropout(self.norm1(attention + x))

        forward = self.feed_forward(x)

        out = self.dropout(self.norm2(forward + x))

        return out

In [12]:
class DecoderBlock(nn.Module):

    def __init__(self, embed_size, heads, dropout):
        super().__init__()

        self.self_attention = MultiHeadAttention(embed_size, heads)

        self.norm = nn.LayerNorm(embed_size)

        self.encoder_attention = MultiHeadAttention(embed_size, heads)

        self.feed_forward = FeedForward(embed_size)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out):

        attention = self.self_attention(x, x, x)

        x = self.dropout(self.norm(attention + x))

        attention2 = self.encoder_attention(enc_out, enc_out, x)

        x = self.dropout(self.norm(attention2 + x))

        forward = self.feed_forward(x)

        out = self.dropout(self.norm(forward + x))

        return out

In [13]:
class Encoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, n_embd)

        self.pos_encoding = PositionalEncoding(n_embd)

        self.layers = nn.ModuleList(
            [EncoderBlock(n_embd, n_head, dropout) for _ in range(n_layer)]
        )

    def forward(self, x):

        out = self.embedding(x)

        out = self.pos_encoding(out)

        for layer in self.layers:
            out = layer(out)

        return out

In [14]:
class Decoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, n_embd)

        self.pos_encoding = PositionalEncoding(n_embd)

        self.layers = nn.ModuleList(
            [DecoderBlock(n_embd, n_head, dropout) for _ in range(n_layer)]
        )

        self.fc = nn.Linear(n_embd, vocab_size)

    def forward(self, x, enc_out):

        x = self.embedding(x)

        x = self.pos_encoding(x)

        for layer in self.layers:
            x = layer(x, enc_out)

        out = self.fc(x)

        return out

In [15]:
class Transformer(nn.Module):

    def __init__(self):
        super().__init__()

        self.encoder = Encoder()

        self.decoder = Decoder()

    def forward(self, src):

        enc_out = self.encoder(src)

        out = self.decoder(src, enc_out)

        return out

In [16]:
model = Transformer().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

criterion = nn.CrossEntropyLoss()

In [17]:
for step in range(500):

    xb, yb = get_batch("train")

    logits = model(xb)

    B,T,C = logits.shape

    loss = criterion(
        logits.view(B*T, C),
        yb.view(B*T)
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 50 == 0:
        print("step", step, "loss", loss.item())

step 0 loss 4.597799301147461
step 50 loss 3.03316330909729
step 100 loss 2.771965980529785
step 150 loss 2.641062021255493
step 200 loss 2.523918628692627
step 250 loss 2.5617105960845947
step 300 loss 2.375664472579956
step 350 loss 2.3598694801330566
step 400 loss 2.137838125228882
step 450 loss 2.083976984024048


In [19]:
def generate(model, max_new_tokens=200):

    context = torch.zeros((1,1), dtype=torch.long).to(device)

    for _ in range(max_new_tokens):

        logits = model(context)

        logits = logits[:,-1,:]

        probs = torch.softmax(logits, dim=-1)

        next_token = torch.multinomial(probs, num_samples=1)

        context = torch.cat([context, next_token], dim=1)

    return decode(context[0].tolist())

In [20]:
print(generate(model))







]

-
P*tettttttrtttttttttt ttite t ttttttttttttttttttttttaettttttttttatttttitetrtttttt tntttttttt tt.
ttt ttttttt otttt t  tNttttttttcttt tto tttttttttt tsttatttttetttttty tt ttttttittptttttt ttt
